# Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.3
!pip install --no-deps trl==0.22.2

In [2]:
!pip install -U torchao --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


# Unsloth

In [3]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "LiquidAI/LFM2.5-230M",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = False, # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    load_in_16bit = True, # [NEW!] Enables 16bit LoRA
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Lfm2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/459M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/303 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/489 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [4]:
FastLanguageModel.for_inference(model)  # enables faster inference mode

messages = [
    {"role": "user", "content": "Translate to Arabic: How are you today?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    temperature=0.7,
    use_cache=True,
)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

['user\nTranslate to Arabic: How are you today?\nassistant\nIn order to translate the phrase "How are you today?" into Arabic, I will follow these steps:\n\n1. Understand the meaning of the phrase.\n2. Translate the words from English to Arabic.\n3. Construct the sentence in Arabic using the translated words and their appropriate context.\n\nStep 1: Understanding the meaning of the phrase:\n"How are you today?" means that the speaker is expressing their current situation or feeling today.\n\nStep 2: Translating the words from English to Arabic:\n- "how" translates to "كيف".\n- "are" translates to "إذا".\n- "you" translates']


In [5]:
FastLanguageModel.for_training(model)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj",
                      "w1", "w2", "w3"],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


# Data Prep

We should use the `LFM` format for conversation style finetunes. LFM renders multi turn conversations like below:

```
<|startoftext|><|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>
```

In [85]:
tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Hello!"},
    {"role" : "assistant", "content" : "Hey there!"}
], tokenize = False)

'<|startoftext|><|im_start|>user\nHello!<|im_end|>\n<|im_start|>assistant\nHey there!<|im_end|>\n'

In [64]:
from datasets import load_dataset
dataset = load_dataset("atlasia/darija-english-combined", split = "train[:10]")

In [65]:
dataset[0]

{'english': 'The result was like lightning. The couple fell one atop of the other, struck down, finding consolation, at last, in death.',
 'darija': 'النتيجة كانت بحال البرق. طاح المزوجين واحد فوق واحد، ولقاو الراحة فالاخير ف الموت.',
 'arabizi': None,
 'source': 'abdeljalilELmajjodi/opus_books_en_fr_darija',
 'script_type': 'darija'}

In [66]:
dataset = dataset.filter(lambda x: x["script_type"] in ["darija", "both"])

Filter:   0%|          | 0/10 [00:00<?, ? examples/s]

In [67]:
len(dataset)

10

In [69]:
def to_conversations(batch):
    convos = []
    for en, darija in zip(batch["english"], batch["darija"]):
        convos.append([{"role": "system", "content": "You are a professional English to Darija translator."},
{"role": "user", "content": en.strip()},
{"role": "assistant", "content": darija.strip()}])
    return {"conversations": convos}

dataset = dataset.map(to_conversations, batched=True, remove_columns=dataset.column_names)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [70]:
dataset

Dataset({
    features: ['conversations'],
    num_rows: 10
})

In case any rows slipped in with different key names

In [71]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/10 [00:00<?, ? examples/s]

In [72]:
def formatting_prompts_func(examples):
    texts = tokenizer.apply_chat_template(
        examples["conversations"],
        tokenize = False,
        add_generation_prompt = False,
    )
    return { "text" : [x.removeprefix(tokenizer.bos_token) for x in texts] }

dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [73]:
dataset = dataset.filter(lambda x: len(x["text"]) <= 2000)

Filter:   0%|          | 0/10 [00:00<?, ? examples/s]

In [74]:
len(dataset)

10

In [75]:
dataset[0]["text"]

'<|im_start|>system\nYou are a professional English to Darija translator.<|im_end|>\n<|im_start|>user\nThe result was like lightning. The couple fell one atop of the other, struck down, finding consolation, at last, in death.<|im_end|>\n<|im_start|>assistant\nالنتيجة كانت بحال البرق. طاح المزوجين واحد فوق واحد، ولقاو الراحة فالاخير ف الموت.<|im_end|>\n'

In [78]:
split_dataset = dataset.train_test_split(test_size=0.1, seed=3407)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(len(train_dataset), len(eval_dataset))

9 1


# Train the model

In [79]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 2,
        packing = True,                        # pack multiple short examples per sequence, huge speedup for translation-length text
        max_seq_length = 2048,                 # keep, but packing uses this efficiently
        warmup_steps = 5,
        num_train_epochs = 3,
        per_device_eval_batch_size = 8,
        eval_strategy = "epoch",
        learning_rate = 2e-4,
        logging_steps = 100,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        group_by_length = True,                 # batches similar-length sequences together, less padding waste
    ),
)

Unsloth: Sample packing skipped (custom data collator detected).


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/9 [00:00<?, ? examples/s]

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/1 [00:00<?, ? examples/s]

In [80]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map (num_proc=2):   0%|          | 0/9 [00:00<?, ? examples/s]

Filter (num_proc=2):   0%|          | 0/9 [00:00<?, ? examples/s]

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


Map (num_proc=1):   0%|          | 0/1 [00:00<?, ? examples/s]

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


Filter (num_proc=1):   0%|          | 0/1 [00:00<?, ? examples/s]

In [82]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

'<|startoftext|><|im_start|>system\nYou are a professional English to Darija translator.<|im_end|>\n<|im_start|>user\nMadame Raquin, feeling the catastrophe near at hand, watched them with piercing, fixed eyes.<|im_end|>\n<|im_start|>assistant\nمدام راكان، اللي حست بلي الكارثة قربات، بقات كتشوف فيهم بعينين حادين وثابتين.<|im_end|>\n'

In [84]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")

'                                              مدام راكان، اللي حست بلي الكارثة قربات، بقات كتشوف فيهم بعينين حادين وثابتين.<|im_end|>\n'

In [32]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
0.48 GB of memory reserved.


In [33]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 615,526 | Num Epochs = 3 | Total steps = 57,708
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 3,096,576 of 232,789,760 (1.33% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [86]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

NameError: name 'trainer_stats' is not defined

In [56]:
messages = [{
    "role": "user",
    "content": "Translate to Darija: A girl can be showing off her art to a boy running by. Final answer: yes",
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 128, # Increase for longer outputs!
    # Recommended Liquid settings!
    temperature = 0.1, top_k = 50, top_p = 0.1, repetition_penalty = 1.05,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

وحدة كا تقدر تبرز فنها لابس ديالها بواحد الرجل اللي كايتمشي. الجواب النهائي: إيه<|im_end|>


In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")